<style>
/* Enable scrolling on long slides */
.reveal .slides section {
    overflow-y: auto !important;
    max-height: 100% !important;
    scrollbar-width: thin;
}
/* Ensure content doesn't get cut off at the bottom */
.reveal .slides section::-webkit-scrollbar {
    width: 6px;
}
.reveal .slides section::-webkit-scrollbar-thumb {
    background: #888;
    border-radius: 10px;
}
</style>

# TLIO: Tight Learned Inertial Odometry

<br>

- Paper: `arXiv:2007.01867v3` / IEEE Robotics and Automation Letters, 2020
- Authors: Wenxin Liu, David Caruso, Eddy Ilg, Jing Dong, Anastasios I. Mourikis, Kostas Daniilidis, Vijay Kumar, Jakob Engel
- Presentation Author: Javier Penuela 

## Introduction to Pedestrian Dead Reckoning

<br>

- Pedestrian dead reckoning (PDR) estimates motion from inertial sensors alone.
- Traditional methods rely on step detection, stride models, and heading estimation.
- Challenges: sensor bias, noise, drift, arbitrary device orientation, and diverse human motion.
- This paper proposes an IMU-only system that avoids step counting by learning short-term displacement priors using a ResNet and physical process modeling.

## Dataset: Collection
<br>

- Collected with a custom rig: Bosch BMI055 IMU mounted on a headset rigidly attached to cameras.
- More than 400 sequences, totaling 60 hours of pedestrian data.
- Activities include walking, standing, kitchen tasks, playing pool, stairs, outdoor uneven terrain, and more.
- Data captured with multiple physical devices (visual data for ground true + IMU data) and over 5 people to cover varied motion patterns and IMU biases.
- Limitations: data excludes looking up or down positions, data does not reflect phone conditions, but smart glasses conditions.

## Dataset: Description
<br>

- Ground truth from a state-of-the-art visual-inertial filter at 1000 Hz.
- Dataset split randomly into 80% training, 10% validation, 10% test. No difference between validation and test datasets is disclosed
- The dataset contains pedestrian trajectories with 3 to 7 minutes of activity per sequence.
- Benchmark comparison uses 3D-RoNIN as a reference baseline. Yet, it ignores RoNIN dataset (100 subjects and 42.7h of data, phone, different possitions)

## Dataset: Preprocessing
<br>

- Training uses overlapping sliding windows of IMU data.
- Each window contains N IMU samples; final choice is `N = 200` for 200 Hz data.
- Data augmentation:
  - random horizontal rotations for yaw invariance (following RoNIN)
  - random sensor bias perturbations
  - random gravity direction perturbations

- Goal: make the network robust to initialization error, bias, and gravity misalignment.

## Dataset: Gravity-Aligned Frame
<br>

- IMU samples are rotated to a local gravity-aligned frame built from the orientation at the beginning of each window.
- Gravity-aligned frame ensures gravity points downward and decouples global yaw from local displacement.
- The network input is therefore invariant to arbitrary heading of the headset.

Method:

Compute a reference array `ig_w: np.array([0, 0, 1.0])` representing gravity pointing "up" in the Z-axis of the world frame.
then they compute the rotation matrix $R$ such that $R a = b$:

- Normalization: It normalizes both the input vector $a$ (the accelerometer reading) and the target vector $b$ (the gravity vector).
- Rotation Axis ($\omega$): It finds the axis of rotation by taking the cross product of the two vectors: $$\omega = \hat{a} \times \hat{b}$$ This vector $\omega$ is perpendicular to the plane formed by $a$ and $b$.




- Rodrigues' Rotation Formula: $$R = I + [\omega]_\times + \frac{1}{1 + \hat{a} \cdot \hat{b}} ([\omega]_\times)^2$$
    $[\omega]_\times$: The skew-symmetric matrix of $\omega$ (computed by the hat(v) helper function -> `np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])`.
   

## System Design Overview
<br>

- Two main components:
  1. Neural network: regresses 3D displacement and uncertainty from IMU segments.
  2. Extended Kalman Filter (EKF): tightly fuses neural displacement measurements with IMU kinematics.

- The network learns a statistical motion prior from data, while the EKF performs model-based propagation.
- IMU data is used twice: directly for state propagation and indirectly as network measurement input. (Data argumentation prevents measurement error propagation to the model)

## Neural Network Description

- Architecture: 1D ResNet18 variant.
- Input: `N x 6` IMU window, where each sample contains accelerometer and gyroscope measurements in gravity-aligned frame.
- Output (one independent FC block (3 layers) per output):
  - $\hat d \in \mathbb{R}^3$: predicted 3D displacement over the window
  - $\hat u \in \mathbb{R}^3$: uncertainty parameters for the displacement covariance

- The network output is treated as a measurement of relative displacement with uncertainty.

## Loss Functions
<br>

- Mean Squared Error (MSE) loss:

$$
L_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^n \| d_i - \hat d_i \|^2
$$

- Maximum Likelihood loss for regressed Gaussian displacement:

$$
L_{\text{ML}} = \frac{1}{n} \sum_{i=1}^n \left( \frac{1}{2} \log \det \Sigma_i + \frac{1}{2} (d_i - \hat d_i)^T \Sigma_i^{-1} (d_i - \hat d_i) \right) + \text{const}
$$

- Covariance parametrization by log-standard deviation:

$$
\Sigma_i = \operatorname{diag}\left(e^{2 u_{x,i}}, e^{2 u_{y,i}}, e^{2 u_{z,i}}\right)
$$

- Best performing training strategy: first train with MSE until stable, then switch to likelihood loss.

## Stochastic Cloning
<br>

- The EKF state includes the current state and a sliding window of past cloned poses.
- A cloned pose is appended whenever a new network measurement is available.
- This allows the filter to use relative displacement measurements between pairs of past states. (i.e. to interpolate `m` piece-wise linear aproximations of the non-linear function between samples `n+k` to `n+k+1` )

- Full state:

$$
\mathbf{X} = (\xi_1, \dots, \xi_m, \mathbf{s})
$$ 

where $\xi_i = ({}^w_i\mathbf{R}_i, {}^w\mathbf{p}_i)$  represents past orientations and positions, and $\mathbf{s}=({}^w_i\mathbf{R}_i, {}^w\mathbf{v}, {}^w\mathbf{p}, \mathbf{b}_g), \mathbf{b}_a$ current position and orientation with dimensions:

$$
\text{dim}(X) = 6m + 15
$$

where `m` is the number of past cloned states.

- Example: with 20 Hz updates and a 1 s window, up to 21 states are maintained.




For the error propagation we apply the error-based filltering method:

Cloned state error: $$\tilde{\xi}_i = (\tilde{\theta}_i, \tilde{\mathbf{p}}_i)$$

Current state error: $$\tilde{\mathbf{s}} = (\tilde{\theta}, \tilde{\mathbf{v}}, \tilde{\mathbf{p}}, \tilde{\mathbf{b}}_g, \tilde{\mathbf{b}}_a)$$

where the rotation error on the $SO(3)$: $\tilde{\theta} = \log_{SO3}(\mathbf{R} \hat{\mathbf{R}}^{-1})$.

## EKF Architecture and Equations
<br>

### Propagation model

Strapdown inertial kinematics with IMU bias and gravity:

$$
R_{k+1} = R_k \exp_{\mathrm{SO(3)}}\left((\omega_k - b_{g,k}) \Delta t\right)
$$
$$
v_{k+1} = v_k + g \Delta t + R_k (a_k - b_{a,k}) \Delta t
$$
$$
p_{k+1} = p_k + v_k \Delta t + \frac{1}{2} \left(g + R_k (a_k - b_{a,k}) \right) \Delta t^2
$$
$$
b_{g,k+1} = b_{g,k} + n_{g,k}, \qquad b_{a,k+1} = b_{a,k} + n_{a,k}
$$



### Linearized propagation
<br>

$$
\tilde s_{k+1} = A_k \tilde s_k + B_k n_k
$$


Standard Propagation: 

$$
P_{k+1} = A_k P_k A_k^T + B_k W B_k^T
$$ 

$$
A_k = \begin{bmatrix} I_{6m} & 0 \\
 0 & A_{s,k} \end{bmatrix}, \quad B_k = \begin{bmatrix} 0 \\
  B_{s,k} \end{bmatrix}
  $$

Propagation with Stochastic Cloning: 

$$
\bar{P}_{k+1} = \bar{A}_k P_k \bar{A}k^T + \bar{B}k \bar{W} \bar{B}k^T
$$ 
$$
\bar{A}k = \begin{bmatrix} I_{6m} & 0 \\ 0 & A_{\xi,k} \\ 0 & A_{s,k} \end{bmatrix}, \quad \bar{B}k = \begin{bmatrix} 0 \\ B_{\xi,k} \\ B_{s,k} \end{bmatrix}
$$




### Measurement model in gravity-aligned frame
<br>

The network predicts a local displacement measurement:

$$
h(X) = R_\gamma^T (p_j^w - p_i^w) = \hat d_{ij} + \eta_{d_{ij}}
$$

where $\eta_{d_{ij}}$ is normally distributed noise, with a zero mean and variance $\hat \Sigma_{ij}$ from the network.

$$\mathbf{H}_{\tilde{\theta}_i} = \frac{\partial h(\mathbf{X})}{\partial \tilde{\theta}_i} = \lfloor \hat{\mathbf{R}}_\gamma^T [{}^w\hat{\mathbf{p}}_j - {}^w\hat{\mathbf{p}}i]_\times \mathbf{H}_z$$

$$\mathbf{H}_{\tilde{\mathbf{\delta p}}_i} = \frac{\partial h(\mathbf{X})}{\partial \delta \tilde{\mathbf{p}}_i} = -\hat{\mathbf{R}}_\gamma^T$$

$$\mathbf{H}_{\tilde{\mathbf{\delta p}}_j} = \frac{\partial h(\mathbf{X})}{\partial \delta \tilde{\mathbf{p}}_j} = \hat{\mathbf{R}}_\gamma^T$$

where
$$\mathbf{H}_z = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 0 & 0 \\ \cos \gamma \tan \beta & \sin \gamma \tan \beta & 1 \end{bmatrix}$$



### Kalman update
<br>

$$
K = P H^T (H P H^T + \hat \Sigma_{ij})^{-1}
$$
$$
X \leftarrow X \oplus K (h(X) - \hat d_{ij})
$$
$$
P \leftarrow (I - K H) P (I - K H)^T + K \hat \Sigma_{ij} K^T
$$

- Measurement Jacobian `H` has nonzero blocks only for the cloned poses `i` and `j`.
- A $\chi^2$ gating test rejects updates with normalized innovation beyond the 99%-tile for 3 DOF.

## Experimental Results
<br>

- The code with the provided environment works well, with the exception of the 3D visualisation module (not compatible with ARM processors)
- The model selection (among the 3 proposed options by the authors) is correct.
- Validation is lower than training in some models, suggesting that the validation set is easier for the model than the training set. Given the small size of the data set, this is likely.




In [6]:
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Forces Plotly to output HTML that nbconvert slides can embed
pio.renderers.default = 'notebook_connected'

def extract_losses(log_dir):
    """Extracts train and validation loss from TensorBoard logs."""
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    
    data = {}
    tags = event_acc.Tags()['scalars']
    target_tags = ['train_loss/avg', 'val_loss/avg']
    
    for tag in target_tags:
        if tag in tags:
            events = event_acc.Scalars(tag)
            steps = [e.step for e in events]
            values = [e.value for e in events]
            
            df_temp = pd.DataFrame({'step': steps, 'value': values})
            df_temp = df_temp.drop_duplicates(subset='step', keep='last').set_index('step')
            data[tag] = df_temp['value']
            
    return pd.DataFrame(data)

# 1. Load data from models directory
models_dir = 'models'
all_data = {}
for model_name in ['resnet', 'resnet_seq', 'tcn']:
    log_path = os.path.join(models_dir, model_name, 'logs')
    if os.path.exists(log_path):
        df = extract_losses(log_path)
        if not df.empty:
            all_data[model_name] = df

# 2. Create interactive Plotly Figure
fig = go.Figure()

# Professional colors for white background
colors = {'resnet': '#1f77b4', 'resnet_seq': '#d62728', 'tcn': '#2ca02c'}

for model_name, df in all_data.items():
    color = colors.get(model_name, '#333333')
    
    if 'train_loss/avg' in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df['train_loss/avg'], mode='lines', 
                                 name=f'{model_name.upper()} Train', line=dict(color=color, width=2)))
    if 'val_loss/avg' in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df['val_loss/avg'], mode='lines', 
                                 name=f'{model_name.upper()} Val', line=dict(color=color, width=2, dash='dot'), opacity=0.7))

# 3. White Theme Styling
fig.update_layout(
    title=dict(text='Model Training Progress (Loss)', font=dict(size=20, color='#222222')),
    xaxis=dict(title='Steps', gridcolor='#EEEEEE', tickfont=dict(color='#444444')),
    yaxis=dict(title='Loss', gridcolor='#EEEEEE', tickfont=dict(color='#444444')),
    paper_bgcolor='white', 
    plot_bgcolor='white', 
    height=400, # Reduced height
    template='plotly_white', 
    hovermode='x unified', 
    margin=dict(l=50, r=50, t=60, b=50),
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, bgcolor='rgba(255,255,255,0.7)')
)

fig.show()


## Metrics
<br>

- Absolute Translation Error (ATE):

$$
\text{ATE} = \sqrt{\frac{1}{n} \sum_{k=1}^n \| p_k^w - \hat p_k^w \|^2}
$$

- Relative Translation Error (RTE(t)): 
$$
\text{RTE}_{\Delta t} = \sqrt{\frac{1}{n} \sum{i=1}^n \left| ({}^w\mathbf{p}_{i+\Delta t} - {}^w\mathbf{p}_i) - \mathbf{R}_\gamma \hat{\mathbf{R}}_\gamma^T ({}^w\hat{\mathbf{p}}_{i+\Delta t} - {}^w\hat{\mathbf{p}}_i) \right|^2}
$$

- Drift Rate (DR): 
$$
\text{DR}(\%) = \frac{| {}^w\mathbf{p}_n - {}^w\hat{\mathbf{p}}_n |}{\text{trajectory-length}} \times 100
$$



- Absolute Yaw Error (AYE): 
$$
\text{AYE} = \sqrt{\frac{1}{n} \sum_{i=1}^n | \psi_i - \hat{\psi}_i |^2}
$$

- Relative Yaw Error (RYE(t)): 
$$
\text{RYE}_{\Delta t} = \sqrt{\frac{1}{n} \sum{i=1}^n | (\psi_{i+t} - \psi_i) - (\hat{\psi}_{i+t} - \hat{\psi}_i) |^2}
$$

- Yaw Drift Rate (Yaw-DR): 
$$
\text{Yaw-DR} = \frac{| \psi_n - \hat{\psi}_n |}{\text{sequence-duration}}
$$

- The paper reports cumulative distribution functions of these metrics over the test set.

## Experimental Results
<br>

### NN benchmarking
- The authors provided a test for the NNs in isolation (use ground true initial position, so the NNs is evaluated in absolute terms).
- We used the provided test by the authors to obtain ATE, RTE and drift. 
- We made the script ##### to benchmark inference time (token by token, full seq, avg single token time)



## RoNIN Benchmark
<br>
- A version of the RoNIN approach (3d-RoNIN ) is the benchmark used in the article.


## System Performance
<br>
- TLIO consistently outperforms 3D-RoNIN on all reported metrics.
- Higher filter update frequency improves ATE and drift, despite measurement correlation.
- The EKF also reduces yaw drift compared to a decoupled attitude filter.
- Key advantages:
  - learned measurement uncertainty improves fusion
  - filter robustness to outliers via `\chi^2` gating
  - no handcrafted step detection or gait model required

## Presenter Comments
<br>
- There is no clarification on what makes validation and test datasets different 
- The authors talk about the world frame, but their code (correctly) does not use the visual data to align the dataset with a true world frame, only to the gravity-aligned frame. 
- Initialisation assumes that the IMU is vertical, --> in real life, that is a poor calibration for the phones.
- In the update part of the filter, it uses Euler angles --> this architecture can present Gimbal lock 
